# पाठ 12 - एजेंट स्क्रैचपैड के साथ चैट इतिहास कमी

यह नोटबुक माइक्रोसॉफ्ट एजेंट फ्रेमवर्क का उपयोग करके लंबी बातचीत में संदर्भ को प्रबंधित करने के तरीके को प्रदर्शित करता है। जैसे-जैसे बातचीत बढ़ती है, टोकन की संख्या बढ़ती है — अंततः मॉडल की संदर्भ विंडो से अधिक हो जाती है। हम इसे **संदर्भ सारांशण पैटर्न** और **एजेंट स्क्रैचपैड** के साथ स्थायी मेमोरी के माध्यम से संबोधित करते हैं।

## आप क्या सीखेंगे:
1. **संदर्भ प्रबंधन क्यों महत्वपूर्ण है**: टोकन सीमाएँ और संदर्भ विंडो को समझना
2. **संदर्भ-सचेत एजेंट्स**: ऐसे एजेंट्स बनाना जो अपनी बातचीत संदर्भ का प्रबंधन करते हैं
3. **संदर्भ सारांशण पैटर्न**: बातचीत इतिहास को संक्षेपित करने के लिए टूल्स का उपयोग
4. **एजेंट स्क्रैचपैड**: स्थायी मेमोरी जो संदर्भ कमी के बाद भी बनी रहती है

## पूर्व आवश्यकताएँ:
- पर्यावरण चर कॉन्फ़िगर किए गए Azure OpenAI सेटअप
- पिछले पाठों से बुनियादी एजेंट अवधारणाओं की समझ


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## संदर्भ प्रबंधन क्यों महत्वपूर्ण है

हर LLM के पास एक सीमित **संदर्भ विंडो** होती है — एक अनुरोध में यह अधिकतम कितने टोकन प्रक्रिया कर सकता है। जैसे-जैसे बहु-चरण वार्तालाप आगे बढ़ता है:

- **टोकन की संख्या रैखिक रूप से बढ़ती है** प्रत्येक उपयोगकर्ता संदेश और सहायक उत्तर के साथ।
- **प्रॉम्प्ट टोकन लागत पर हावी होते हैं** क्योंकि पूरी इतिहास हर दौर में फिर से भेजा जाता है।
- अंततः वार्तालाप **संदर्भ विंडो को पार कर जाता है** और मॉडल या तो इसे छोटा करता है या त्रुटि देता है।

### संदर्भ प्रबंधन के लिए रणनीतियाँ

| रणनीति | यह कैसे काम करता है | समझौता |
|---|---|---|
| **छंटाई** | सबसे पुराने संदेशों को हटाना | प्रारंभिक संदर्भ खो जाता है |
| **सारांशकरण** | पुराने संदेशों को एक सारांश में संक्षिप्त करना | कुछ विवरण खो जाता है, लेकिन मुख्य बिंदु बनाए रहते हैं |
| **स्क्रैचपैड / बाहरी मेमोरी** | मुख्य तथ्य वार्तालाप के बाहर संग्रहित करना | टूल कॉल की जरूरत होती है, लेकिन किसी भी कमी से बचाता है |

इस नोटबुक में हम **सारांशकरण** को एक **स्क्रैचपैड टूल** के साथ मिलाते हैं ताकि एजेंट उस समय भी निरंतरता बनाए रख सके जब वार्तालाप इतिहास संक्षिप्त किया गया हो।


## एक संदर्भ-सचेत एजेंट बनाना


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## लंबे संवाद का अनुकरण करना

आइए एक बहु-चरणीय बातचीत के माध्यम से चलते हैं ताकि यह देखा जा सके कि संदर्भ कैसे जमा होता है। एजेंट को मुख्य विवरण (पसंद, बजट, यात्रा तिथियां) को चरणों के दौरान बनाए रखना चाहिए और निरंतरता प्रदर्शित करनी चाहिए।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ध्यान दें कि एजेंट पहले के संवादों से संदर्भ बनाए रखता है — उसे जापान, सुशी, मंदिरों, फोटोग्राफी, $3000 बजट, अकेले यात्रा, और अप्रैल के समय के बारे में जानकारी है। एक छोटी बातचीत में यह अच्छी तरह काम करता है, लेकिन जैसे-जैसे बातचीत बढ़ती है, पूरा इतिहास पुनः भेजना महंगा हो जाता है।

चलिए बातचीत को और अधिक चरणों के साथ जारी रखते हैं ताकि संदर्भ संचय को देखा जा सके:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## संदर्भ सारांश पैटर्न

जैसे-जैसे बातचीत बढ़ती है, हम संचित संदर्भ को संक्षिप्त स्वरूप में समेटने के लिए एक **सारांश उपकरण** का उपयोग कर सकते हैं। एजेंट इस उपकरण को मुख्य प्राथमिकताओं को रिकॉर्ड करने के लिए कॉल करता है ताकि पुराने संदेश हटाए जाने पर भी आवश्यक जानकारी सुरक्षित रहे।

यह पैटर्न अधिक परिष्कृत इतिहास संक्षेपण के लिए आधार है:
1. एजेंट बातचीत से मुख्य तथ्यों की पहचान करता है
2. इसे स्थायी करने के लिए सारांश उपकरण को कॉल करता है
3. पुराने संदेश सुरक्षित रूप से हटा दिए जा सकते हैं क्योंकि सारांश महत्वपूर्ण बातों को पकड़ता है

नीचे हम एक `summarize_preferences` उपकरण परिभाषित करते हैं जिसे एजेंट सीखे गए मुख्य बिंदुओं का संक्षिप्त सारांश रिकॉर्ड करने के लिए कॉल कर सकता है।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके लंबे समय तक चलने वाली एजेंट बातचीत में संदर्भ को प्रबंधित करना सीखा:

### प्रमुख अवधारणाएँ
- **संदर्भ विंडो सीमित होती हैं** — बातचीत के इतिहास में प्रत्येक टोकन की लागत होती है और यह सीमा में गिना जाता है।
- **सारांश उपकरण** एजेंट को संचित संदर्भ को संक्षिप्त सारांशों में बदलने देते हैं, जिससे आवश्यक जानकारी बनाए रखते हुए टोकन उपयोग कम होता है।
- **एजेंट स्क्रैचपैड** लगातार बाहरी स्मृति प्रदान करता है जो किसी भी बातचीत में कटौती के बाद भी जीवित रहता है।

### आपने क्या बनाया
- एक **संदर्भ-संवेदनशील एजेंट** जो मल्टी-टर्न बातचीत में निरंतरता बनाए रखता है
- एक **सारांश उपकरण** (`summarize_preferences`) जो उपयोगकर्ता की प्रमुख जानकारी को संक्षिप्त रूप में रिकॉर्ड करता है
- एक **मल्टी-टर्न बातचीत** जो संदर्भ संरक्षण और परिवर्तन प्रबंधन को दर्शाती है

### वास्तविक दुनिया में उपयोग
- **ग्राहक सेवा बॉट्स**: लंबे सपोर्ट सत्रों के दौरान प्राथमिकताएँ याद रखें
- **व्यक्तिगत सहायक**: संदर्भ फिर से समझाए बिना चल रहे परियोजनाओं को ट्रैक करें
- **शैक्षिक ट्यूटर**: कई बातचीत के माध्यम से छात्र की प्रगति बनाए रखें

### आगे के कदम
- फ़ाइल-आधारित स्थायित्व के साथ पूर्ण स्क्रैचपैड उपकरण लागू करें
- सारांश के बाद स्वचालित इतिहास कटौती जोड़ें
- सेमांटिक मेमोरी खोज के लिए वेक्टर डेटाबेस के साथ संयोजन करें
- ऐसे एजेंट बनाएं जो कई दिनों बाद भी पूरी संदर्भ के साथ बातचीत फिर से शुरू कर सकें


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
इस दस्तावेज़ का अनुवाद एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान रखें कि स्वचालित अनुवाद में त्रुटियाँ या गलतियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सिफारिश की जाती है। हम इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या व्याख्या के लिए जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
